### Preprocessing: Data Cleaning & Feature Engineering

Goal:
Load and clean our raw electricity and weather data, align timezones, handle missing values, and engineer crucial predictors like cyclical time features (sines/cosines). Save it in data/processed/hourly_merged.csv`

Why did we use it?
Raw real world data is messy. This step ensures our data is mathematically sound and structured in a way that machine learning algorithms can actually understand and process.

In [1]:
import os
import numpy as np
import pandas as pd

BASE_DIR  = os.path.abspath('..')
EWZ_PATH  = os.path.join(BASE_DIR, 'data', 'raw', 'ewz',
                         'ewz_stromabgabe_netzebenen_stadt_zuerich.csv')
METEO_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'meteo')
OUT_PATH  = os.path.join(BASE_DIR, 'data', 'processed', 'hourly_merged.csv')

DATE_START   = '2022-01-01'
DATE_END     = '2025-12-31 23:00'
METEO_PARAMS = ['T', 'Hr', 'StrGlo', 'RainDur', 'WVv', 'p']

## 1. EWZ - Electricity Demand

In [2]:
df_ewz = pd.read_csv(EWZ_PATH)
df_ewz['Timestamp'] = pd.to_datetime(df_ewz['Timestamp'], utc=True)
df_ewz = df_ewz.set_index('Timestamp')

# total demand = NE5 + NE7, resample from 15-min to hourly mean
df_ewz['demand'] = df_ewz['Value_NE5'] + df_ewz['Value_NE7']
df_ewz = df_ewz[['demand']].resample('h').mean()
df_ewz.index.name = 'datetime_utc'
df_ewz = df_ewz.loc[DATE_START:DATE_END]

print(f'{len(df_ewz)} hourly rows')
df_ewz.head(3)

35064 hourly rows


,demand
datetime_utc,
2022-01-01 00:00:00+00:00,60787.89700
2022-01-01 01:00:00+00:00,59943.68675
2022-01-01 02:00:00+00:00,57751.22200


## 2. Meteo - Weather Data

In [3]:
# load all yearly files in range
chunks = []
for fname in sorted(os.listdir(METEO_DIR)):
    if not fname.endswith('.csv'):
        continue
    year = int(fname.split('_')[-1].replace('.csv', ''))
    if year < 2022 or year > 2025:
        continue
    chunks.append(pd.read_csv(os.path.join(METEO_DIR, fname), na_values=['NA']))

meteo_raw = pd.concat(chunks, ignore_index=True)

# timestamps include the UTC offset so pandas converts automatically
meteo_raw['datetime_utc'] = pd.to_datetime(meteo_raw['Datum'], utc=True)
meteo_raw = meteo_raw[meteo_raw['Parameter'].isin(METEO_PARAMS)].copy()

print(f'{meteo_raw["Standort"].nunique()} stations, {len(meteo_raw):,} rows')

4 stations, 613,656 rows


In [4]:
# pick the station with fewest missing values across all parameters
raw_range = meteo_raw[
    (meteo_raw['datetime_utc'] >= DATE_START) &
    (meteo_raw['datetime_utc'] <= DATE_END)
]
completeness = (
    raw_range.groupby(['Standort', 'Parameter'])['Wert']
    .apply(lambda s: s.notna().sum())
    .unstack('Parameter')
    .reindex(columns=METEO_PARAMS, fill_value=0)
)
# use the min so a station missing one parameter entirely scores low
completeness['score'] = completeness[METEO_PARAMS].min(axis=1)
best_station = completeness['score'].idxmax()

print(f'Selected station: {best_station}')
completeness[METEO_PARAMS]

Selected station: Zch_Stampfenbachstrasse


Parameter,T,Hr,StrGlo,RainDur,WVv,p
Standort,,,,,,
Zch_Heubeeribüel,17537.0,17537.0,NaN,NaN,NaN,17537.0
Zch_Rosengartenstrasse,33925.0,33925.0,NaN,33912.0,33553.0,34018.0
Zch_Schimmelstrasse,34806.0,34530.0,NaN,34654.0,34823.0,34202.0
Zch_Stampfenbachstrasse,35017.0,35017.0,35029.0,35034.0,35042.0,35035.0


In [5]:
# pivot to wide format: one row per hour, one column per parameter
df_meteo = meteo_raw[meteo_raw['Standort'] == best_station].copy()
df_meteo = df_meteo.pivot_table(index='datetime_utc', columns='Parameter',
                                 values='Wert', aggfunc='mean')
df_meteo.columns.name = None
df_meteo.index.name   = 'datetime_utc'
df_meteo = df_meteo[METEO_PARAMS].loc[DATE_START:DATE_END]

print(f'{len(df_meteo)} hourly rows')
df_meteo.head(3)

35045 hourly rows


,T,Hr,StrGlo,RainDur,WVv,p
datetime_utc,,,,,,
2022-01-01 00:00:00+00:00,6.78,83.34,0.02,0.0,0.90,977.07
2022-01-01 01:00:00+00:00,6.42,84.66,0.03,0.0,0.70,977.14
2022-01-01 02:00:00+00:00,6.15,84.61,0.02,0.0,0.49,977.38


## 3. Merge

In [6]:
# both indexes are UTC so the join lines up correctly
df = df_ewz.join(df_meteo, how='inner')
print(f'Merged: {len(df)} rows  |  {df.shape[1]} columns')
df.head(3)

Merged: 35045 rows  |  7 columns


,demand,T,Hr,StrGlo,RainDur,WVv,p
datetime_utc,,,,,,,
2022-01-01 00:00:00+00:00,60787.89700,6.78,83.34,0.02,0.0,0.90,977.07
2022-01-01 01:00:00+00:00,59943.68675,6.42,84.66,0.03,0.0,0.70,977.14
2022-01-01 02:00:00+00:00,57751.22200,6.15,84.61,0.02,0.0,0.49,977.38


## 4. Feature Engineering

In [7]:
# use local Zurich time for the time features
local = df.index.tz_convert('Europe/Zurich')

df['hour']       = local.hour
df['dayofweek']  = local.dayofweek   # 0 = Monday
df['month']      = local.month
df['is_weekend'] = (local.dayofweek >= 5).astype(int)

season_map = {12: 'winter', 1: 'winter', 2: 'winter',
               3: 'spring', 4: 'spring', 5: 'spring',
               6: 'summer', 7: 'summer', 8: 'summer',
               9: 'autumn', 10: 'autumn', 11: 'autumn'}
df['season'] = df['month'].map(season_map)

# cyclic encoding so the model sees hour 23 and hour 0 as close
df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df[['hour', 'dayofweek', 'month', 'is_weekend', 'season',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos']].head(3)

,hour,dayofweek,month,is_weekend,season,hour_sin,hour_cos,month_sin,month_cos
datetime_utc,,,,,,,,,
2022-01-01 00:00:00+00:00,1,5,1,1,winter,0.258819,0.965926,0.5,0.866025
2022-01-01 01:00:00+00:00,2,5,1,1,winter,0.500000,0.866025,0.5,0.866025
2022-01-01 02:00:00+00:00,3,5,1,1,winter,0.707107,0.707107,0.5,0.866025


## 5. Model Targets

In [8]:
# demand_log for the linear model
df['demand_log'] = np.log(df['demand'])

# is_peak: 1 if demand exceeds the 90th percentile
p90 = df['demand'].quantile(0.90)
df['is_peak'] = (df['demand'] > p90).astype(int)
print(f'p90 threshold: {p90:.1f}  |  peak hours: {df["is_peak"].sum()} ({df["is_peak"].mean()*100:.1f}%)')

# peak_hours_per_day for the Poisson model
date_col = df.index.tz_convert('Europe/Zurich').normalize().tz_localize(None)
df['date'] = date_col
daily = df.groupby('date')['is_peak'].sum().rename('peak_hours_per_day').astype(int)
df = df.join(daily, on='date').drop(columns='date')

df[['demand_log', 'is_peak', 'peak_hours_per_day']].head(3)

p90 threshold: 95305.4  |  peak hours: 3505 (10.0%)


,demand_log,is_peak,peak_hours_per_day
datetime_utc,,,
2022-01-01 00:00:00+00:00,11.015146,0,0
2022-01-01 01:00:00+00:00,11.001161,0,0
2022-01-01 02:00:00+00:00,10.963900,0,0


## 6. Save & Summary

In [9]:
assert len(df) <= 100_000, f'Too many rows: {len(df)}'

df.index = df.index.strftime('%Y-%m-%dT%H:%M:%SZ')
df.index.name = 'datetime_utc'
df.to_csv(OUT_PATH)
print(f'Saved to {OUT_PATH}')

Saved to C:\Users\paula\Desktop\ml1-electricity-demand-zurich\data\processed\hourly_merged.csv


In [10]:
print(f'Shape: {df.shape}')
print(f'\nColumns:\n{list(df.columns)}')
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.describe(include=[np.number]).round(2)

Shape: (35045, 19)

Columns:
['demand', 'T', 'Hr', 'StrGlo', 'RainDur', 'WVv', 'p', 'hour', 'dayofweek', 'month', 'is_weekend', 'season', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'demand_log', 'is_peak', 'peak_hours_per_day']

Missing values:
T          28
Hr         28
StrGlo     16
RainDur    11
WVv         3
p          10
dtype: int64


,demand,T,Hr,StrGlo,RainDur,WVv,p,hour,dayofweek,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos,demand_log,is_peak,peak_hours_per_day
count,35045.00,35017.00,35017.00,35029.00,35034.00,35042.00,35035.00,35045.00,35045.0,35045.00,35045.00,35045.00,35045.00,35045.00,35045.00,35045.00,35045.0,35045.00
mean,74295.02,12.87,68.78,141.14,5.80,1.59,966.70,11.50,3.0,6.53,0.29,0.00,-0.00,-0.01,-0.00,11.19,0.1,2.40
std,15555.92,7.87,16.76,229.07,15.59,1.08,7.54,6.92,2.0,3.45,0.45,0.71,0.71,0.71,0.71,0.21,0.3,3.98
min,46140.74,-5.51,17.91,0.01,0.00,0.00,928.84,0.00,0.0,1.00,0.00,-1.00,-1.00,-1.00,-1.00,10.74,0.0,0.00
25%,60349.08,6.90,57.63,0.02,0.00,0.78,962.78,6.00,1.0,4.00,0.00,-0.71,-0.71,-0.87,-0.87,11.01,0.0,0.00
50%,73374.98,12.35,72.39,4.46,0.00,1.42,966.95,12.00,3.0,7.00,0.00,0.00,-0.00,-0.00,-0.00,11.20,0.0,0.00
75%,87781.51,18.76,82.11,198.14,0.00,2.16,971.19,17.00,5.0,10.00,1.00,0.71,0.71,0.50,0.87,11.38,0.0,3.00
max,110250.54,35.80,101.37,1041.18,60.00,9.89,990.27,23.00,6.0,12.00,1.00,1.00,1.00,1.00,1.00,11.61,1.0,13.00
